Read parsed PDF, scan for important words and phrases related to food and nutrition, and build four small dictionaries (nutrients, ingredients, techniques, and guidelines) that you’ll use to train or guide your Named Entity Recognition model later.

In [15]:
import json, re, unicodedata
from collections import defaultdict, Counter
from pathlib import Path

RAW = Path("data/raw_text.jsonl")  

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

vitamin_rx = re.compile(r"\bvitamin[s]?\s+[A-Z](?:\d+)?\b", re.I)
minerals_list = ["calcium","iron","zinc","iodine","selenium","potassium","magnesium","phosphorus"]
macro_micro_list = ["fibre","fiber","saturated fat","unsaturated fat","omega-3","omega-6","protein","carbohydrate","sodium","salt","sugars","free sugar","added sugar"]
food_groups = ["cereals","grains","whole grain","vegetables","berries","fruits","legumes","pulses","nuts","seeds","fish","seafood","red meat","processed meat","poultry","milk","dairy","eggs","fats","oils","beverages","alcohol","potatoes","rye","oats","barley"]
tech_terms = ["smoking","salting","nitrite","curing","fermentation","pasteurisation","pasteurization","boiling","frying","baking","grilling","fortification","iodised salt","iodized salt"]
guideline_verbs = ["limit","reduce","increase","prefer","choose","eat more","aim to","replace","avoid"]

gaz = defaultdict(Counter)

with open(RAW, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        section = obj.get("section")
        path = obj.get("section_path")
        text = obj.get("text","")
        section_lc = norm(section) if isinstance(section, str) else ""
        path_lc = [norm(x) for x in path] if isinstance(path, list) else ([section_lc] if section_lc else [])

        # ---------- Nutrients ----------
        if any("vitamin" in x or "mineral" in x for x in path_lc) or "vitamins and minerals" in norm(text):
            for m in vitamin_rx.findall(text):
                gaz["nutrient"][m.lower()] += 1
            for m in minerals_list + macro_micro_list:
                if re.search(rf"\b{re.escape(m)}\b", text, re.I):
                    gaz["nutrient"][m.lower()] += 1
        # Fallback: always scan text for vitamin/mineral cues
        for m in vitamin_rx.findall(text):
            gaz["nutrient"][m.lower()] += 1
        for m in minerals_list:
            if re.search(rf"\b{re.escape(m)}\b", text, re.I):
                gaz["nutrient"][m.lower()] += 1

        # ---------- Ingredients ----------
        if any("recommended food" in x or "food choices" in x for x in path_lc) or "recommended food choices" in norm(text):
            for term in food_groups:
                if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                    gaz["ingredient"][term] += 1
        for term in food_groups:
            if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                gaz["ingredient"][term] += 1

        # ---------- Techniques ----------
        if any("glossary" in x or "safety" in x or "processing" in x for x in path_lc) or "processed" in norm(text):
            for t in tech_terms:
                if re.search(rf"\b{re.escape(t)}\b", text, re.I):
                    gaz["technique"][t] += 1
        for t in tech_terms:
            if re.search(rf"\b{re.escape(t)}\b", text, re.I):
                gaz["technique"][t] += 1

        # ---------- Guidelines ----------
        if any("recommendation" in x for x in path_lc) or "recommendation" in norm(text) or "recommendations" in norm(text):
            for v in guideline_verbs:
                for m in re.findall(rf"\b{v}\s+[a-z][a-z\- ]{{2,30}}", text, re.I):
                    gaz["guideline"][m.strip().lower()] += 1

# keep items that occur >=2 times to filter noise
seed_vocab = {k: sorted([term for term, c in cnts.items() if c >= 2]) for k, cnts in gaz.items()}

print("Seeds summary (counts):", {k: len(v) for k, v in seed_vocab.items()})
for k, terms in seed_vocab.items():
    print(f"\n{k.upper()} ({len(terms)}):")
    for t in terms[:50]:
        print(" •", t)

with open("data/seed_vocabularies.json", "w", encoding="utf-8") as out:
    json.dump(seed_vocab, out, ensure_ascii=False, indent=2)
print("\n✅ Saved seed_vocabularies.json")


Seeds summary (counts): {'ingredient': 25, 'guideline': 3, 'technique': 7, 'nutrient': 22}

INGREDIENT (25):
 • alcohol
 • barley
 • berries
 • beverages
 • cereals
 • dairy
 • eggs
 • fats
 • fish
 • fruits
 • grains
 • legumes
 • milk
 • nuts
 • oats
 • oils
 • potatoes
 • poultry
 • processed meat
 • red meat
 • rye
 • seafood
 • seeds
 • vegetables
 • whole grain

GUIDELINE (3):
 • aim to influence the population
 • reduce the adverse environmental impac
 • reduce the environmental impact of foo

TECHNIQUE (7):
 • baking
 • fermentation
 • fortification
 • iodised salt
 • nitrite
 • salting
 • smoking

NUTRIENT (22):
 • calcium
 • fibre
 • iodine
 • iron
 • magnesium
 • phosphorus
 • potassium
 • protein
 • salt
 • saturated fat
 • selenium
 • vitamin a
 • vitamin a2
 • vitamin b12
 • vitamin b6
 • vitamin c
 • vitamin d
 • vitamin d3
 • vitamin e
 • vitamin k
 • vitamins a
 • zinc

✅ Saved seed_vocabularies.json


In [16]:
import json, re, unicodedata
from collections import defaultdict, Counter
from pathlib import Path

RAW = Path("data/raw_text.jsonl") 

def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

# --- Existing domain patterns ---
vitamin_rx = re.compile(r"\bvitamin[s]?\s+[A-Z](?:\d+)?\b", re.I)
minerals_list = ["calcium","iron","zinc","iodine","selenium","potassium","magnesium","phosphorus"]
macro_micro_list = ["fibre","fiber","saturated fat","unsaturated fat","omega-3","omega-6","protein","carbohydrate","sodium","salt","sugars","free sugar","added sugar"]
food_groups = ["cereals","grains","whole grain","vegetables","berries","fruits","legumes","pulses","nuts","seeds","fish","seafood","red meat","processed meat","poultry","milk","dairy","eggs","fats","oils","beverages","alcohol","potatoes","rye","oats","barley"]
tech_terms = ["smoking","salting","nitrite","curing","fermentation","pasteurisation","pasteurization","boiling","frying","baking","grilling","fortification","iodised salt","iodized salt"]
guideline_verbs = ["limit","reduce","increase","prefer","choose","eat more","aim to","replace","avoid"]

# --- NEW: healthOutcome + environmentImpact patterns ---
# Keep these lists simple and conservative; we only mine from your PDF text.
health_outcome_terms = [
    # diseases / endpoints
    "cardiovascular disease","ischemic heart disease","stroke","type 2 diabetes","obesity","overweight",
    "hypertension","high blood pressure","blood pressure","cholesterol","ldl cholesterol","hdl cholesterol",
    "triglycerides","cancer","colorectal cancer","all-cause mortality","mortality","morbidity","inflammation",
    "insulin resistance","metabolic syndrome"
]

# phrases that often introduce risk associations
risk_intro = r"(risk|risk of|associated with|linked to|higher odds of|increases|reduces)"

environment_terms = [
    "greenhouse gas emissions","ghg emissions","carbon footprint","climate impact","environmental impact",
    "land use","water use","water footprint","nitrogen footprint","phosphorus footprint","biodiversity",
    "biodiversity loss","ecological footprint","emissions","food system emissions","sustainability"
]

gaz = defaultdict(Counter)

with open(RAW, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        section = obj.get("section") or ""
        text = obj.get("text","")
        section_lc = norm(section)
        text_norm = norm(text)

        # ---------- Nutrients ----------
        if "vitamin" in section_lc or "mineral" in section_lc or "vitamins and minerals" in text_norm:
            for m in vitamin_rx.findall(text):
                gaz["nutrient"][m.lower()] += 1
            for m in minerals_list + macro_micro_list:
                if re.search(rf"\b{re.escape(m)}\b", text, re.I):
                    gaz["nutrient"][m.lower()] += 1
        # Fallback scan
        for m in vitamin_rx.findall(text):
            gaz["nutrient"][m.lower()] += 1
        for m in minerals_list:
            if re.search(rf"\b{re.escape(m)}\b", text, re.I):
                gaz["nutrient"][m.lower()] += 1

        # ---------- Ingredients ----------
        if "recommended food" in section_lc or "food choices" in section_lc or "recommended food choices" in text_norm:
            for term in food_groups:
                if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                    gaz["ingredient"][term] += 1
        for term in food_groups:
            if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                gaz["ingredient"][term] += 1

        # ---------- Techniques ----------
        if any(k in section_lc for k in ["glossary","safety","processing"]) or "processed" in text_norm:
            for t in tech_terms:
                if re.search(rf"\b{re.escape(t)}\b", text, re.I):
                    gaz["technique"][t] += 1
        for t in tech_terms:
            if re.search(rf"\b{re.escape(t)}\b", text, re.I):
                gaz["technique"][t] += 1

        # ---------- Guidelines ----------
        if "recommendation" in section_lc or "recommendations" in section_lc or "recommendation" in text_norm:
            for v in guideline_verbs:
                for m in re.findall(rf"\b{v}\s+[a-z][a-z\- ]{{2,30}}", text, re.I):
                    gaz["guideline"][m.strip().lower()] += 1

        # ---------- NEW: Health Outcomes ----------
        # Section cues
        if any(k in section_lc for k in ["health", "disease", "risk", "mortality", "morbidity"]):
            for term in health_outcome_terms:
                if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                    gaz["healthOutcome"][term] += 1
        # Text cues
        for term in health_outcome_terms:
            if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                gaz["healthOutcome"][term] += 1
        # Simple risk-pattern harvest (e.g., "risk of stroke", "reduces blood pressure")
        risk_pattern = re.compile(rf"{risk_intro}\s+([a-z][a-z\- ]{{2,40}})", re.I)
        for match in risk_pattern.finditer(text_norm):
            # match.groups() returns (risk_word, phrase)
            phrase = match.group(1).strip()
            if 4 <= len(phrase) <= 40 and not phrase.endswith((" of"," to"," in")):
                if any(w in phrase for w in ["disease","diabetes","obesity","pressure","cholesterol","mortality","cancer","stroke"]):
                    gaz["healthOutcome"][phrase] += 1

        # ---------- NEW: Environmental Impacts ----------
        if any(k in section_lc for k in ["environment", "climate", "sustainab", "emission", "footprint", "biodiversity"]):
            for term in environment_terms:
                if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                    gaz["environmentImpact"][term] += 1
        for term in environment_terms:
            if re.search(rf"\b{re.escape(term)}\b", text, re.I):
                gaz["environmentImpact"][term] += 1
        # Abbreviations
        if re.search(r"\bghg\b", text_norm):
            gaz["environmentImpact"]["ghg emissions"] += 1

# Keep items seen >=2 times (lower to 1 if your doc is sparse)
threshold = 2
seed_vocab = {k: sorted([term for term, c in cnts.items() if c >= threshold]) for k, cnts in gaz.items()}

print("Seeds summary (counts):", {k: len(v) for k, v in seed_vocab.items()})
for k, terms in seed_vocab.items():
    print(f"\n{k.upper()} ({len(terms)}):")
    for t in terms[:60]:
        print(" •", t)

with open("data/seed_vocabularies.json", "w", encoding="utf-8") as out:
    json.dump(seed_vocab, out, ensure_ascii=False, indent=2)
print("\n✅ Saved seed_vocabularies.json with healthOutcome and environmentImpact")


Seeds summary (counts): {'ingredient': 25, 'guideline': 3, 'environmentImpact': 10, 'technique': 7, 'nutrient': 22, 'healthOutcome': 16}

INGREDIENT (25):
 • alcohol
 • barley
 • berries
 • beverages
 • cereals
 • dairy
 • eggs
 • fats
 • fish
 • fruits
 • grains
 • legumes
 • milk
 • nuts
 • oats
 • oils
 • potatoes
 • poultry
 • processed meat
 • red meat
 • rye
 • seafood
 • seeds
 • vegetables
 • whole grain

GUIDELINE (3):
 • aim to influence the population
 • reduce the adverse environmental impac
 • reduce the environmental impact of foo

ENVIRONMENTIMPACT (10):
 • biodiversity
 • biodiversity loss
 • carbon footprint
 • climate impact
 • emissions
 • environmental impact
 • greenhouse gas emissions
 • land use
 • sustainability
 • water footprint

TECHNIQUE (7):
 • baking
 • fermentation
 • fortification
 • iodised salt
 • nitrite
 • salting
 • smoking

NUTRIENT (22):
 • calcium
 • fibre
 • iodine
 • iron
 • magnesium
 • phosphorus
 • potassium
 • protein
 • salt
 • saturated f

Try to improve guidelines

In [17]:
import json, re, unicodedata
from pathlib import Path
from collections import Counter

RAW = Path("data/raw_text.jsonl")
SEEDS = Path("data/seed_vocabularies.json")

# --- helpers ---
def norm(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def clean_obj(obj: str) -> str:
    o = obj.strip()
    o = re.sub(r"(?i)\b(intake|consumption|use|amount|servings)\s+of\s+", "", o)
    o = re.sub(r"(?i)\b(the|this|that|their|your|our)\b", "", o).strip()
    o = re.sub(r"[().,;:]+$", "", o).strip()
    o = re.sub(r"\s+", " ", o)
    return norm(o)

# Map synonymous verbs to canonical forms
VERB_MAP = {
    "reduce": "reduce", "decrease": "reduce", "cut": "reduce", "cut down": "reduce", "lower": "reduce",
    "limit": "limit", "restrict": "limit",
    "avoid": "avoid",
    "increase": "increase", "raise": "increase", "boost": "increase", "eat": "increase", "consume": "increase",
    "prefer": "prefer", "choose": "prefer", "prioritise": "prefer", "prioritize": "prefer",
    "replace": "replace", "swap": "replace", "substitute": "replace",
}

# Compile patterns
# 1) Bullet/imperative lines (•, -, or plain)
imperative = re.compile(
    r"(?im)^(?:\s*[•\-\u2022]\s*)?(?P<verb>limit|reduce|decrease|avoid|restrict|increase|prefer|choose|eat|consume|replace|swap|substitute)\s+(?P<obj>[A-Za-z][A-Za-z\- ]{2,80})$"
)

# 2) Modal recommendations
modal = re.compile(
    r"(?i)\b(should|ought to|are advised to|are encouraged to|are recommended to|it is recommended to|recommended to|aim to|seek to|try to)\s+"
    r"(?P<verb>limit|reduce|decrease|avoid|restrict|increase|prefer|choose|eat|consume|replace|swap|substitute)\s+(?P<obj>[A-Za-z][A-Za-z\- ]{2,80})"
)

# 3) “No more than … of X” → limit X
limit_qty = re.compile(
    r"(?i)\b(no more than|not more than|maximum of|no greater than)\s+\d+[^\s]{0,6}\s+(?P<obj>salt|sodium|sugar|free sugar|added sugar|alcohol|red meat|processed meat|saturated fat|meat)\b"
)

# 4) Prefer X over Y  / Replace X with Y  → keep both forms
prefer_over = re.compile(r"(?i)\b(prefer|choose)\s+(?P<a>[A-Za-z][A-Za-z\- ]{2,60})\s+(?:over|rather than)\s+(?P<b>[A-Za-z][A-Za-z\- ]{2,60})")
replace_with = re.compile(r"(?i)\b(replace|swap|substitute)\s+(?P<a>[A-Za-z][A-Za-z\- ]{2,60})\s+(?:with|by)\s+(?P<b>[A-Za-z][A-Za-z\- ]{2,60})")

guidelines = Counter()

with RAW.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        text = obj.get("text", "")
        # 1) imperative bullets/lines
        for m in imperative.finditer(text):
            verb = VERB_MAP.get(m.group("verb").lower(), m.group("verb").lower())
            obj_str = clean_obj(m.group("obj"))
            if obj_str:
                guidelines[f"{verb} {obj_str}"] += 1
        # 2) modal forms
        for m in modal.finditer(text):
            verb = VERB_MAP.get(m.group("verb").lower(), m.group("verb").lower())
            obj_str = clean_obj(m.group("obj"))
            if obj_str:
                guidelines[f"{verb} {obj_str}"] += 1
        # 3) quantitative limits → "limit X"
        for m in limit_qty.finditer(text):
            obj_str = clean_obj(m.group("obj"))
            if obj_str:
                guidelines[f"limit {obj_str}"] += 1
        # 4) prefer/replace relations
        for m in prefer_over.finditer(text):
            a = clean_obj(m.group("a")); b = clean_obj(m.group("b"))
            if a and b:
                guidelines[f"prefer {a} over {b}"] += 1
        for m in replace_with.finditer(text):
            a = clean_obj(m.group("a")); b = clean_obj(m.group("b"))
            if a and b:
                guidelines[f"replace {a} with {b}"] += 1

# Keep items that occur >= 2 times (lower to 1 if needed)
THRESH = 1
harvested = sorted([g for g, c in guidelines.items() if c >= THRESH])

print(f"Harvested guideline candidates: {len(harvested)}")
for g in harvested[:20]:
    print(" •", g)

# Update seed_vocabularies.json
seeds = json.loads(SEEDS.read_text(encoding="utf-8")) if SEEDS.exists() else {}
seeds["guideline"] = harvested  # overwrite guideline bucket
SEEDS.write_text(json.dumps(seeds, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"\n✅ Updated {SEEDS} with {len(harvested)} guideline phrases.")


Harvested guideline candidates: 5
 • increase accordingly
 • increase supply
 • limit caffeine intake to a maximum of
 • prefer sources of unsaturated fat such as vegetable oils
 • reduce saturated fat

✅ Updated data/seed_vocabularies.json with 5 guideline phrases.


Try to improve guidelines in another way

In [24]:
import json, re, unicodedata
from pathlib import Path

RAW = Path("data/raw_text.jsonl")  # <- use the uploaded file

def norm_spaces(s:str)->str:
    s = unicodedata.normalize("NFKC", s)
    s = s.replace("\u2013","-").replace("\u2014","-")
    s = re.sub(r"\s+", " ", s).strip()
    return s

# join all text
docs = []
with RAW.open(encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        txt = obj.get("text","")
        if txt:
            docs.append(norm_spaces(txt))
corpus = " ".join(docs)

# sentence split (simple + keeps units intact)
SENT_SPLIT = re.compile(r'(?<!\b[A-Z])[.!?]\s+(?=[A-Z])')
sentences = [s.strip() for s in SENT_SPLIT.split(corpus) if len(s.strip())>0]
len(sentences)


1330

In [25]:
# canonical action verbs (map surface → normalized)
VERB_MAP = {
    "limit":"limit","reduce":"limit","decrease":"limit","restrict":"limit","avoid":"avoid",
    "prefer":"prefer","choose":"prefer","opt for":"prefer","favour":"prefer","favor":"prefer",
    "eat":"include","include":"include","add":"include","increase":"increase",
    "replace":"replace","swap":"replace","substitute":"replace"
}

# object cleaner
def clean_obj(o:str)->str:
    o = o.strip()
    # drop generic “intake/consumption of …”
    o = re.sub(r"(?i)\b(intake|consumption|use|amount|servings?)\s+of\s+", "", o)
    # trim trailing qualifiers like “per day/week”
    o = re.sub(r"(?i)\s+(?:per|/)\s*(day|week|month)\b.*$", "", o).strip(" ,.;:")
    return o

# quantities / units / frequencies
NUM = r"(?:\d+(?:[\.,]\d+)?)"
RANGE = rf"(?:{NUM}\s*(?:–|-|to)\s*{NUM})"
VAL = rf"(?:{RANGE}|{NUM})"
UNIT = r"(?:g|mg|µg|mcg|μg|E%|kcal|kJ|portion(?:s)?|serving(?:s)?)"
PER = r"(?:\s*(?:per|/)\s*(?:day|week))?"
LIMIT_Q = rf"(?:to\s+(?:a\s+maximum\s+of\s+)?{VAL}\s*{UNIT}{PER})"
ATLEAST_Q = rf"(?:at\s+least\s+{VAL}\s*{UNIT}{PER})"
NOMORE_Q = rf"(?:no\s+more\s+than\s+{VAL}\s*{UNIT}{PER})"
NOTMORE_Q = rf"(?:not\s+more\s+than\s+{VAL}\s*{UNIT}{PER})"

# 1) Imperative pattern
imperative = re.compile(
    rf"(?i)\b(?P<verb>limit|reduce|decrease|avoid|restrict|increase|prefer|choose|eat|include|replace|swap|substitute)\s+"
    rf"(?P<obj>[A-Za-z][A-Za-z\- ]{{2,100}}?)"
    rf"(?:\s+(?P<qual>{LIMIT_Q}|{ATLEAST_Q}|{NOMORE_Q}|{NOTMORE_Q}))?\b"
)

# 2) Modal recommendations
modal = re.compile(
    rf"(?i)\b(?:should|ought to|are advised to|are encouraged to|it is recommended to|is recommended to|recommended to|aim to|seek to|try to)\s+"
    rf"(?P<verb>limit|reduce|decrease|avoid|restrict|increase|prefer|choose|eat|include|replace|swap|substitute)\s+"
    rf"(?P<obj>[A-Za-z][A-Za-z\- ]{{2,100}}?)"
    rf"(?:\s+(?P<qual>{LIMIT_Q}|{ATLEAST_Q}|{NOMORE_Q}|{NOTMORE_Q}))?\b"
)

# 3) “No more than / at least … of X”
quant_of = re.compile(
    rf"(?i)\b(?P<qual>{NOMORE_Q}|{NOTMORE_Q}|{ATLEAST_Q})\s+of\s+(?P<obj>[A-Za-z][A-Za-z\- ]{{2,100}}?)\b"
)

# 4) Replace X with Y (capture both)
replace_with = re.compile(
    rf"(?i)\b(?:replace|swap|substitute)\s+(?P<obj>[A-Za-z][A-Za-z\- ]{{2,100}}?)\s+(?:with|for)\s+(?P<alt>[A-Za-z][A-Za-z\- ]{{2,100}}?)\b"
)

def normalize_action(v):
    return VERB_MAP.get(v.lower(), v.lower())


In [26]:
from collections import defaultdict

def harvest_guidelines(sents):
    out = []
    for s in sents:
        # 1) replace-with rules first (produce two forms)
        for m in replace_with.finditer(s):
            obj = clean_obj(m.group("obj"))
            alt = clean_obj(m.group("alt"))
            out.append({"action":"replace", "object":obj, "qualifier":f"with {alt}", "source":s})

        # 2) imperative
        for m in imperative.finditer(s):
            action = normalize_action(m.group("verb"))
            obj = clean_obj(m.group("obj"))
            qual = m.group("qual")
            out.append({"action":action, "object":obj, "qualifier":norm_spaces(qual) if qual else "", "source":s})

        # 3) modal
        for m in modal.finditer(s):
            action = normalize_action(m.group("verb"))
            obj = clean_obj(m.group("obj"))
            qual = m.group("qual")
            out.append({"action":action, "object":obj, "qualifier":norm_spaces(qual) if qual else "", "source":s})

        # 4) quantified “of X”
        for m in quant_of.finditer(s):
            out.append({"action":"limit/include", "object":clean_obj(m.group("obj")),
                        "qualifier":norm_spaces(m.group("qual")), "source":s})

    # Canonical string for de-duplication
    seen = set()
    uniq = []
    for g in out:
        key = (g["action"], g["object"].lower(), g["qualifier"].lower())
        if key not in seen and len(g["object"])>=3:
            seen.add(key)
            uniq.append(g)
    return uniq

guidelines = harvest_guidelines(sentences)
len(guidelines), guidelines[:5]


(86,
 [{'action': 'prefer',
   'object': 'sources',
   'qualifier': '',
   'source': 'In addition, diets should prefer sources of unsaturated fat such as vegetable oils, fish and nuts, and limit the use of salt'},
  {'action': 'limit',
   'object': 'the',
   'qualifier': '',
   'source': 'In addition, diets should prefer sources of unsaturated fat such as vegetable oils, fish and nuts, and limit the use of salt'},
  {'action': 'increase',
   'object': 'the',
   'qualifier': '',
   'source': 'A central objective of the nutrition recommendations is to increase the supply and availability of health-promoting foods to all population groups'},
  {'action': 'limit',
   'object': 'energy',
   'qualifier': '',
   'source': 'Water and fibre reduce energy density'},
  {'action': 'include',
   'object': 'all',
   'qualifier': '',
   'source': 'Whole grains include all the edible parts of the grain, including the bran and germ'}])

In [27]:
# pretty formatter for the entity text
def fmt_guideline(g):
    a, o, q = g["action"], g["object"], g["qualifier"]
    if a == "replace" and q.startswith("with "):
        return f"replace {o} {q}"
    return f"{a} {o}" + (f" {q}" if q else "")

# export
cleaned = sorted({fmt_guideline(g) for g in guidelines})
dietary_guidelines = [{"type":"dietaryGuideline", "name":dg} for dg in cleaned]

print(f"Harvested dietary guidelines: {len(dietary_guidelines)}")
for s in dietary_guidelines[:20]:
    print(" •", s["name"])

# (optionally) update your seed_vocabularies.json
import json
SEEDS = Path("data/seed_vocabularies.json")
seeds = json.loads(SEEDS.read_text(encoding="utf-8")) if SEEDS.exists() else {}
seeds["guideline"] = [s["name"] for s in dietary_guidelines]
SEEDS.write_text(json.dumps(seeds, ensure_ascii=False, indent=2), encoding="utf-8")


Harvested dietary guidelines: 86
 • avoid butter
 • avoid heavy
 • avoid high
 • avoid potentially
 • avoid products
 • avoid the
 • include a variety
 • include all
 • include bacteria
 • include butter
 • include carbohydrates
 • include cereals
 • include children
 • include coffee
 • include criteria
 • include cruciferous
 • include dairy
 • include edible
 • include elsewhere
 • include fatigue


3474

Using YAKE to improve the seeds:
YAKE can surface domain phrases that seed lists missed, which can then be promoted to entities (when they match ontology classes) and re-run NER.

In [4]:
%pip install yake

Note: you may need to restart the kernel to use updated packages.


In [5]:
import json, unicodedata, re
from collections import Counter
from pathlib import Path
import yake

RAW = Path("data/raw_text.jsonl")

def norm(s):
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

kw = yake.KeywordExtractor(lan="en", n=3, top=15)  # adjust lan if needed
corpus_counts = Counter()

with RAW.open("r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        text = obj.get("text", "")
        if not isinstance(text, str):
            text = str(text) if text is not None else ""
        if not text.strip():
            continue  # skip empty chunks

        try:
            phrases = kw.extract_keywords(text)
        except Exception:
            # YAKE can occasionally choke on odd unicode; skip this chunk
            continue

        for phrase, score in phrases:
            if not phrase:       # None or empty
                continue
            p = norm(phrase)
            if not p:            # empty after normalization
                continue
            if 4 <= len(p) <= 60:
                corpus_counts[p] += 1

print("Total unique phrases:", len(corpus_counts))
# (optional) peek at top items
for ph, c in corpus_counts.most_common(20):
    print(c, ph)


Total unique phrases: 1141
47 food
29 intake
26 health
20 sustainable health
17 consumption
15 vitamin
15 recommended
14 nutrition
14 products
12 health from food
11 directions
10 recommendations
10 diet
10 finland
10 women
10 years
9 nutrition recommendations
9 sustainable
9 recommended intake
9 exclusive breastfeeding


In [6]:
# --- 2) Load existing seeds; build fast lookups for novelty filtering ---
import json

SEEDS = Path("data/seed_vocabularies.json")

seeds = json.loads(SEEDS.read_text(encoding="utf-8")) if SEEDS.exists() else {}
for b in ("nutrient","ingredient","technique","guideline","healthOutcome","environmentImpact"):
    seeds.setdefault(b, [])

seed_set = {p for terms in seeds.values() for p in terms}  # flat set (lowercased already if you stored lower)


In [7]:
# --- 3) Classify candidate phrases to ontology classes (rules + cues) ---
# Section/name cues you already use elsewhere (lowercase class names)
SECTION_CUES = {
    "nutrient": ["vitamin","mineral","micronutrient"],
    "ingredient": ["recommended food","food choices","foods"],
    "technique": ["glossary","safety","processing","processed"],
    "dietaryGuideline": ["recommendation"],
    "healthOutcome": ["health","disease","risk","mortality","morbidity","blood pressure","cholesterol"],
    "environmentImpact": ["environment","climate","sustainab","emission","footprint","biodiversity"],
}

# Simple lexical hints for typing (keep conservative)
vitamin_rx = re.compile(r"\bvitamin\s+[a-z0-9]+\b")
nutrient_terms = {"calcium","iron","iodine","selenium","zinc","magnesium","phosphorus","potassium","fibre","fiber","protein","carbohydrate","sodium","salt","omega-3","omega-6","saturated fat","unsaturated fat"}
food_terms = {"fish","red meat","processed meat","poultry","whole grain","vegetables","fruits","berries","legumes","nuts","seeds","milk","dairy","eggs","beverages","alcohol","grains","cereals","rye","oats","barley","potatoes","oils","fats","seafood"}
tech_terms = {"fortification","smoking","salting","fermentation","pasteurisation","pasteurization","baking","frying","grilling","iodised salt","iodized salt","nitrite"}
health_markers = {"obesity","overweight","hypertension","blood pressure","cholesterol","ldl cholesterol","hdl cholesterol","triglycerides","mortality","morbidity","cancer","colorectal cancer","stroke","type 2 diabetes","cardiovascular disease","inflammation"}
env_terms = {"greenhouse gas emissions","ghg emissions","emissions","carbon footprint","environmental impact","climate impact","land use","water use","water footprint","nitrogen footprint","biodiversity","biodiversity loss","sustainability"}

def classify(phrase: str):
    p = phrase
    if vitamin_rx.search(p) or p in nutrient_terms:
        return "nutrient"
    if p in food_terms:
        return "ingredient"
    if p in tech_terms:
        return "technique"
    if p in health_markers:
        return "healthOutcome"
    if p in env_terms:
        return "environmentImpact"
    # weak heuristics: 2-word phrases ending with 'use'/'footprint' => envImpact
    if re.search(r"(footprint|use)$", p):
        return "environmentImpact"
    return None  # leave guidelines to a separate pass; they are verb-led phrases


In [8]:
# --- 4) Select novel, classifiable phrases and update seeds ---
MIN_DOC_FREQ = 2  # raise to be stricter
added = defaultdict(list)

for phrase, freq in corpus_counts.most_common():
    if freq < MIN_DOC_FREQ:
        break  # counts sorted? (Counter.most_common() is sorted)
    if phrase in seed_set:
        continue
    cls = classify(phrase)
    if cls:
        added[cls].append(phrase)

# Merge into seeds (dedupe + sort)
for cls, items in added.items():
    seeds[cls] = sorted({*seeds[cls], *items})

SEEDS.write_text(json.dumps(seeds, ensure_ascii=False, indent=2), encoding="utf-8")
print("Added by class:", {k: len(v) for k,v in added.items()})
print("Updated seed_vocabularies.json")


Added by class: {'nutrient': 2}
Updated seed_vocabularies.json


Use spaCy for NER pipeline

In [28]:
# If spaCy isn't installed in your environment, run this once:
%pip install -q spacy


Note: you may need to restart the kernel to use updated packages.


1) Load ontology + seeds, prepare label map

In [29]:
import json, yaml, re, unicodedata, math
from pathlib import Path
from collections import defaultdict, Counter

RAW_PATH   = Path("data/raw_text.jsonl")
SEEDS_PATH = Path("data/seed_vocabularies.json")
ONTO_PATH  = Path("data/ontology.yaml")

# --- ontology essentials ---
onto = yaml.safe_load(ONTO_PATH.read_text(encoding="utf-8"))
BASE_URI = onto.get("base_uri", "http://example.org/food#")

# Allow these classes (extend if ontology has others)
VALID_CLASSES = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "dietaryGuideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

# Map our seed keys -> ontology classes
LABEL_MAP = {
    "nutrient": "nutrient",
    "ingredient": "ingredient",
    "technique": "technique",
    "guideline": "dietaryGuideline",
    "healthOutcome": "healthOutcome",
    "environmentImpact": "environmentImpact",
}

seeds = json.loads(SEEDS_PATH.read_text(encoding="utf-8"))
for k in list(seeds.keys()):
    if k not in LABEL_MAP:
        print("Skipping unknown seed group:", k)

print("Loaded seeds:", {k: len(v) for k,v in seeds.items()})
print("Base URI:", BASE_URI)


Loaded seeds: {'ingredient': 25, 'guideline': 86, 'environmentImpact': 10, 'technique': 7, 'nutrient': 22, 'healthOutcome': 16}
Base URI: http://example.org/food#


2) Build a spaCy EntityRuler from seeds (no external training)

In [30]:
import spacy
from spacy.pipeline import EntityRuler

# Load a small blank model (no internet needed)
nlp = spacy.blank("en")

# Create ruler
ruler = nlp.add_pipe("entity_ruler", config={"overwrite_ents": True})

patterns = []
def term_to_pattern(term: str):
    # simple, case-insensitive token pattern
    # handle hyphens & spaces without overfitting
    tokens = re.split(r"[\s\-]+", term.strip())
    return [{"LOWER": t.lower()} for t in tokens if t]

for group, terms in seeds.items():
    label = LABEL_MAP.get(group)
    if not label: 
        continue
    for term in terms:
        if not term.strip():
            continue
        patterns.append({"label": label, "pattern": term_to_pattern(term), "id": term})

ruler.add_patterns(patterns)
print(f"EntityRuler loaded with {len(patterns)} patterns across {len(seeds)} groups.")


EntityRuler loaded with 166 patterns across 6 groups.


3) Run NER over chunks and collect mentions (with spans + provenance)

In [31]:
def norm_text(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).casefold().strip()
    s = re.sub(r"\s+", " ", s)
    return s

def context_window(txt, start, end, win=80):
    a = max(0, start-win)
    b = min(len(txt), end+win)
    return txt[a:b]

# optional section-based typing cues if your JSONL only has `section`
SECTION_CUES = {
    "nutrient": ["vitamin", "mineral", "micronutrient"],
    "ingredient": ["recommended food", "food choices", "foods"],
    "technique": ["glossary", "safety", "processing", "processed"],
    "dietaryGuideline": ["recommendation"],
    "healthOutcome": ["health", "disease", "risk", "mortality", "morbidity", "blood pressure", "cholesterol"],
    "environmentImpact": ["environment", "climate", "sustainab", "emission", "footprint", "biodiversity"],
}

mentions = []  # flat list of detected mentions

with RAW_PATH.open("r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        obj = json.loads(line)
        text = obj.get("text", "")
        section = obj.get("section", "")
        doc = nlp(text)

        for ent in doc.ents:
            # Prefer ruler label; if missing, try section cues (rare)
            label = ent.label_
            if not label or label not in VALID_CLASSES.values():
                sec = norm_text(section)
                for L, cues in SECTION_CUES.items():
                    if any(c in sec for c in cues):
                        label = L
                        break
            if not label:
                continue

            # record mention with provenance
            m = {
                "chunk_id": obj.get("id") or f"raw:{i:06d}",
                "page": obj.get("page"),
                "section": section,
                "type": label,
                "text_span": [int(ent.start_char), int(ent.end_char)],
                "surface": text[ent.start_char:ent.end_char],
                "context": context_window(text, ent.start_char, ent.end_char),
            }
            mentions.append(m)

print(f"Collected {len(mentions)} mentions.")
# quick peek
for x in mentions[:5]:
    print(x)


Collected 1618 mentions.
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'nutrient', 'text_span': [556, 560], 'surface': 'salt', 'context': 's have improved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake. More plant- based diet'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'nutrient', 'text_span': [565, 578], 'surface': 'saturated fat', 'context': 'proved in recent decades, but the challenges that remain are excessive salt and saturated fat intakes and insufficient fibre intake. More plant- based diets, i.e. an increas'}
{'chunk_id': 'raw:000001', 'page': 5, 'section': 'Preface', 'type': 'nutrient', 'text_span': [604, 609], 'surface': 'fibre', 'context': 'enges that remain are excessive salt and saturated fat intakes and insufficient fibre intake. More plant- based diets, i.e. an increase in the proportion of whole gr'}
{'chunk_id': 'raw:000001', 'page': 5, 'section

4) Canonicalization & alias/coref merge → entities.jsonl

In [32]:
import difflib
import re, string, hashlib, json

# Safe slug for URIs
def slug(s: str) -> str:
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s).strip("_")
    return s or hashlib.md5(s.encode()).hexdigest()[:8]

def normalize_alias(s: str) -> str:
    s = norm_text(s)
    s = s.replace("-", " ")
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

# terms we should NOT merge even if similar
DO_NOT_MERGE = {
    ("added sugar","free sugar"),
    ("red meat","processed meat"),
    ("land use","water use"),
    ("ldl cholesterol","hdl cholesterol"),
}

def looks_equivalent(a: str, b: str, threshold=0.92) -> bool:
    if a == b:
        return True
    # token-set similarity
    aset, bset = set(a.split()), set(b.split())
    inter = aset & bset
    if (len(inter) / max(1, min(len(aset), len(bset)))) >= 0.8:
        return True
    # difflib ratio
    if difflib.SequenceMatcher(a=a, b=b).ratio() >= threshold:
        return True
    return False

def protected_pair(a, b):
    ap, bp = tuple(sorted((a,b)))
    return (ap, bp) in DO_NOT_MERGE

# 1) bucket mentions by type first
by_type = defaultdict(list)
for m in mentions:
    by_type[m["type"]].append(m)

entities = {}  # id -> entity record

for etype, mlist in by_type.items():
    # 2) within each type, cluster by normalized surface
    clusters = []  # list of dicts: {"rep": str, "norms": set(), "mentions": []}
    for m in mlist:
        surf_norm = normalize_alias(m["surface"])
        assigned = False
        for cl in clusters:
            if not protected_pair(surf_norm, cl["rep"]) and looks_equivalent(surf_norm, cl["rep"]):
                cl["norms"].add(surf_norm)
                cl["mentions"].append(m)
                assigned = True
                break
        if not assigned:
            clusters.append({"rep": surf_norm, "norms": {surf_norm}, "mentions":[m]})

    # 3) build canonical entity per cluster
    for cl in clusters:
        # canonical label: most common capitalized surface among mentions
        cap_counts = Counter([m["surface"] for m in cl["mentions"]])
        label = max(cap_counts, key=cap_counts.get)

        eid = BASE_URI + slug(label)
        if eid in entities:
            # merge if same slug got created for some reason
            entities[eid]["aliases"].update(cl["norms"])
            entities[eid]["mentions"].extend(cl["mentions"])
        else:
            entities[eid] = {
                "id": eid,
                "type": etype,
                "label": label,
                "aliases": set(cl["norms"]),
                "source": "sustainable-health-from-food_web.pdf",
                "mentions": list(cl["mentions"]),
            }

# finalize: sort mentions, convert alias set to list
for e in entities.values():
    e["aliases"] = sorted(e["aliases"])
    e["mentions"].sort(key=lambda m: (m.get("page") or 0, m["text_span"][0]))

# write JSONL
out_path = Path("data/entities.jsonl")
with out_path.open("w", encoding="utf-8") as out:
    for e in entities.values():
        out.write(json.dumps(e, ensure_ascii=False) + "\n")

print(f"Wrote {len(entities)} canonical entities -> {out_path}")


Wrote 138 canonical entities -> data/entities.jsonl


Print the unique entity type and label

In [33]:
import json

# Path to your entities file
path = "data/entities.jsonl"

# Load all entities (one JSON object per line)
entities = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            entities.append(json.loads(line))

# Print summary
print(f"Total entities: {len(entities)}\n")

# Sort by type and label
entities_sorted = sorted(entities, key=lambda e: (e["type"], e["label"].lower()))

for e in entities_sorted:
    print(f"{e['type']:<20} | {e['label']}")


Total entities: 138

dietaryGuideline     | Avoid butter
dietaryGuideline     | avoid heavy
dietaryGuideline     | Avoid high
dietaryGuideline     | avoid potentially
dietaryGuideline     | Avoid products
dietaryGuideline     | avoid the
dietaryGuideline     | include a variety
dietaryGuideline     | include all
dietaryGuideline     | include bacteria
dietaryGuideline     | include butter
dietaryGuideline     | include carbohydrates
dietaryGuideline     | include children
dietaryGuideline     | include coffee
dietaryGuideline     | include criteria
dietaryGuideline     | include cruciferous
dietaryGuideline     | include dairy
dietaryGuideline     | include edible
dietaryGuideline     | include fatigue
dietaryGuideline     | include fish
dietaryGuideline     | include foods
dietaryGuideline     | include macrocytic
dietaryGuideline     | include meat
dietaryGuideline     | include milk
dietaryGuideline     | include more
dietaryGuideline     | include nutrient
dietaryGuideline     | in

Lowecase all entities names (called "label")

In [34]:
import json
from pathlib import Path

# Path to your JSONL file
input_path = Path("data/entities.jsonl")
output_path = Path("data/entities_lowercased.jsonl")

count = 0
with input_path.open("r", encoding="utf-8") as infile, \
     output_path.open("w", encoding="utf-8") as outfile:
    for line in infile:
        if not line.strip():
            continue
        obj = json.loads(line)
        if "label" in obj and isinstance(obj["label"], str):
            obj["label"] = obj["label"].lower().strip()
        json.dump(obj, outfile, ensure_ascii=False)
        outfile.write("\n")
        count += 1

print(f"✅ Lowercased {count} entities and saved to {output_path}")


✅ Lowercased 138 entities and saved to data/entities_lowercased.jsonl


Print label and type of every entity

In [36]:
import json

# Path to your entities file
path = "data/entities_lowercased.jsonl"

# Load all entities (one JSON object per line)
entities = []
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            entities.append(json.loads(line))

# Print summary
print(f"Total entities: {len(entities)}\n")

# Sort by type and label
entities_sorted = sorted(entities, key=lambda e: (e["type"], e["label"].lower()))

for e in entities_sorted:
    print(f"{e['type']:<20} | {e['label']}")


Total entities: 138

dietaryGuideline     | avoid butter
dietaryGuideline     | avoid heavy
dietaryGuideline     | avoid high
dietaryGuideline     | avoid potentially
dietaryGuideline     | avoid products
dietaryGuideline     | avoid the
dietaryGuideline     | include a variety
dietaryGuideline     | include all
dietaryGuideline     | include bacteria
dietaryGuideline     | include butter
dietaryGuideline     | include carbohydrates
dietaryGuideline     | include children
dietaryGuideline     | include coffee
dietaryGuideline     | include criteria
dietaryGuideline     | include cruciferous
dietaryGuideline     | include dairy
dietaryGuideline     | include edible
dietaryGuideline     | include fatigue
dietaryGuideline     | include fish
dietaryGuideline     | include foods
dietaryGuideline     | include macrocytic
dietaryGuideline     | include meat
dietaryGuideline     | include milk
dietaryGuideline     | include more
dietaryGuideline     | include nutrient
dietaryGuideline     | in